In [1]:
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
import xgboost as xgb
from xgboost import XGBRegressor

In [2]:
import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [3]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelXG/ModelXG.json")

In [4]:
def SurrogateModelOfReality(n_ci, n_it):
    y_pred = loaded_model.predict(np.array([[n_ci],[n_it]]).T)[0]
    return np.float64(y_pred)

In [5]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [6]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=3)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}
            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 0 =========================================
14.392064094543457

Trial 1 =========================================
17.13752555847168

Trial 2 =========================================
16.545217514038086

Trial 3 =========================================
15.68480396270752



/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/User

Trial 4 =========================================
16.036523818969727

Trial 5 =========================================
13.579638481140137



/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/User

...

Trial 87 =========================================
16.35771369934082

Trial 88 =========================================
16.624534606933594

Trial 89 =========================================
15.184355735778809



/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 90 =========================================
16.672197341918945

Trial 91 =========================================
16.462270736694336

Trial 92 =========================================
16.545217514038086

Trial 93 =========================================
13.758429527282715

Trial 94 =========================================
16.35771369934082

Trial 95 =========================================
18.626216888427734

Trial 96 =========================================
16.349613189697266



/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 97 =========================================
16.545217514038086

Trial 98 =========================================
15.394299507141113

Trial 99 =========================================
13.758429527282715

[14.39206409 17.13752556 16.54521751 15.68480396 16.03652382 13.57963848
 16.28254128 15.39429951 16.62453461 16.39373589 16.62453461 13.89389229
 17.34830475 16.54521751 15.62542725 14.28152466 16.62453461 17.05820084
 16.46227074 16.67219734 17.13752556 16.54521751 16.39373589 16.39373589
 15.68480396 13.75842953 16.26813889 17.05820084 16.67219734 16.6074276
 16.54521751 15.26731396 16.03652382 13.89389229 16.62582397 16.3577137
 13.88541508 14.21930599 13.57963848 15.72569561 19.13921356 17.12041473
 15.39429951 17.34830475 17.05820084 16.54521751 16.6074276  15.35331726
 17.12041473 15.26731396 16.03652382 14.34629345 15.68480396 16.3577137
 14.29863071 16.30029869 17.34830475 16.97524834 15.39429951 16.67219734
 16.03652382 16.3577137  14.35105228 16.54521751 16.39373589

In [7]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 19.13921356201172
Avg = 15.979169759750366
Std = 1.1542504557678548


In [8]:
print(y_max_arr.tolist())

[14.392064094543457, 17.13752555847168, 16.545217514038086, 15.68480396270752, 16.036523818969727, 13.579638481140137, 16.282541275024414, 15.394299507141113, 16.624534606933594, 16.393735885620117, 16.624534606933594, 13.893892288208008, 17.348304748535156, 16.545217514038086, 15.62542724609375, 14.281524658203125, 16.624534606933594, 17.05820083618164, 16.462270736694336, 16.672197341918945, 17.13752555847168, 16.545217514038086, 16.393735885620117, 16.393735885620117, 15.68480396270752, 13.758429527282715, 16.268138885498047, 17.05820083618164, 16.672197341918945, 16.6074275970459, 16.545217514038086, 15.267313957214355, 16.036523818969727, 13.893892288208008, 16.625823974609375, 16.35771369934082, 13.885415077209473, 14.219305992126465, 13.579638481140137, 15.725695610046387, 19.13921356201172, 17.12041473388672, 15.394299507141113, 17.348304748535156, 17.05820083618164, 16.545217514038086, 16.6074275970459, 15.353317260742188, 17.12041473388672, 15.267313957214355, 16.036523818969

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelXG/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelXG/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    16.357714
1    15.938946
2    17.058201
3    17.058201
4    15.171805
..         ...
995  18.626217
996  16.349613
997  16.545218
998  15.394300
999  13.758430

[1000 rows x 1 columns]
